# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayushmanlohani/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Rule: Prioritize pages that are both stale and still visible in search. A page gets a higher review score when it has gone longer without an update while still receiving meaningful impressions.

Reason codes:
- stale_visible_page
- visible_recent_page
- low_visibility_page*

In [7]:
import pandas as pd
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

signal_df = con.sql(f"""
    SELECT
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE
            WHEN gsc_impressions > 0
            THEN 100.0 * gsc_clicks / gsc_impressions
            ELSE NULL
        END AS ctr
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
      AND gsc_avg_position > 0
""").df()

print("Rows:", len(signal_df))
print(signal_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3447872
            content_hash_id  gsc_impressions  gsc_clicks  gsc_avg_position  \
0  content_b7e512995f79d5a6               20           0          3.350000   
1  content_7a105f548d9c6916              125           1          4.928000   
2  content_905aa32a0230694e                7           0          4.000000   
3  content_a3ea9792f793ec72               11           0          2.272727   
4  content_36c36abc7650d7af              239           1          7.347280   

       ctr  
0  0.00000  
1  0.80000  
2  0.00000  
3  0.00000  
4  0.41841  


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I score pages by combining search visibility with weaker CTR and worse average position. Higher-scoring pages are ranked earlier for review. The score uses simple, transparent conditions with no fitted weights.

In [8]:
# Simple thresholds from this March 2026 slice
impression_threshold = signal_df["gsc_impressions"].median()
ctr_threshold = signal_df["ctr"].median()
position_threshold = signal_df["gsc_avg_position"].median()

signal_df["visible"] = (
    signal_df["gsc_impressions"] >= impression_threshold
).astype(int)

signal_df["low_ctr"] = (
    signal_df["ctr"] <= ctr_threshold
).astype(int)

signal_df["poor_position"] = (
    signal_df["gsc_avg_position"] >= position_threshold
).astype(int)

# Transparent score: visibility × number of weak signals × impressions
signal_df["score"] = (
    signal_df["visible"]
    * (signal_df["low_ctr"] + signal_df["poor_position"])
    * signal_df["gsc_impressions"]
)

def get_reason(row):
    reasons = []

    if row["low_ctr"]:
        reasons.append("low_ctr")

    if row["poor_position"]:
        reasons.append("poor_position")

    if not reasons:
        return "no_flag"

    return "+".join(reasons)

signal_df["reason_code"] = signal_df.apply(get_reason, axis=1)

signal_df["action"] = np.where(
    signal_df["score"] > 0,
    "review",
    "no_action"
)

# Rank highest score first
queue = (
    signal_df
    .sort_values(["score", "gsc_impressions"], ascending=False)
    .reset_index(drop=True)
)

queue["rank"] = queue.index + 1

# Keep the ranked output
queue = queue[
    [
        "rank",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position"
    ]
]

# Save required CSV
import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows ranked:", len(queue))
print("\nTop 10:")
display(queue.head(10))

Saved: work/outputs/baseline_action_score.csv
Rows ranked: 3447872

Top 10:


,rank,content_hash_id,score,reason_code,action,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
0,1,content_945d6ff91386c817,74736,low_ctr+poor_position,review,37368,0,0.000000,8.613948
1,2,content_fec55986a1868d62,33383,low_ctr,review,33383,0,0.000000,0.181500
2,3,content_44f34c0a90047651,32958,low_ctr,review,32958,0,0.000000,0.132532
3,4,content_fec55986a1868d62,31472,low_ctr,review,31472,0,0.000000,0.083407
4,5,content_9c057b66c30a3abb,28973,low_ctr,review,28973,0,0.000000,0.000311
5,6,content_9c057b66c30a3abb,28947,low_ctr,review,28947,0,0.000000,0.002245
6,7,content_34a70fea29d15f24,27410,low_ctr,review,27410,0,0.000000,3.129405
7,8,content_66288edeb93b7c4f,24577,poor_position,review,24577,66,0.268544,10.794239
8,9,content_9c057b66c30a3abb,24233,low_ctr,review,24233,0,0.000000,0.317996
9,10,content_66288edeb93b7c4f,23542,poor_position,review,23542,165,0.700875,11.112140


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 ranked pages. The action is based only on the observed signals used by the baseline. Confidence is higher when multiple signals agree. A pick could be wrong if the observed signals do not reflect the page's actual search situation.

In [9]:
top20 = queue.head(20).copy()

top20["confidence_note"] = np.where(
    top20["reason_code"].str.contains("+", regex=False),
    "Higher: both signals agree",
    "Lower: only one signal triggered"
)

top20["what_would_make_it_wrong"] = (
    "The observed signal may not reflect the page's actual search situation."
)

review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review)

,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_945d6ff91386c817,review,low_ctr+poor_position,Higher: both signals agree,The observed signal may not reflect the page's...
1,2,content_fec55986a1868d62,review,low_ctr,Lower: only one signal triggered,The observed signal may not reflect the page's...
2,3,content_44f34c0a90047651,review,low_ctr,Lower: only one signal triggered,The observed signal may not reflect the page's...
3,4,content_fec55986a1868d62,review,low_ctr,Lower: only one signal triggered,The observed signal may not reflect the page's...
4,5,content_9c057b66c30a3abb,review,low_ctr,Lower: only one signal triggered,The observed signal may not reflect the page's...
5,6,content_9c057b66c30a3abb,review,low_ctr,Lower: only one signal triggered,The observed signal may not reflect the page's...
6,7,content_34a70fea29d15f24,review,low_ctr,Lower: only one signal triggered,The observed signal may not reflect the page's...
7,8,content_66288edeb93b7c4f,review,poor_position,Lower: only one signal triggered,The observed signal may not reflect the page's...
8,9,content_9c057b66c30a3abb,review,low_ctr,Lower: only one signal triggered,The observed signal may not reflect the page's...
9,10,content_66288edeb93b7c4f,review,poor_position,Lower: only one signal triggered,The observed signal may not reflect the page's...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some top picks are weaker when only one signal triggered, because there is less supporting evidence. The baseline uses only March 2026 observed search fields and does not use product flags, labels, or future-window data.

In [10]:
# Weak picks: top-20 entries supported by only one signal
weak_picks = top20[
    ~top20["reason_code"].str.contains("+", regex=False)
].copy()

weak_picks["why_weak"] = (
    "Only one observed signal triggered, so the evidence is limited."
)

print("WEAK PICKS")
display(
    weak_picks[
        ["rank", "content_hash_id", "reason_code", "score", "why_weak"]
    ]
)

# Leakage check: inspect the actual warehouse columns
schema_cols = con.sql(
    f"DESCRIBE SELECT * FROM {MAR}"
).df()["column_name"].tolist()

forbidden_terms = [
    "label",
    "trend",
    "flag",
    "future",
    "product"
]

suspicious_cols = [
    col for col in schema_cols
    if any(term in col.lower() for term in forbidden_terms)
]

print("\nLEAKAGE CHECK")
print("Suspicious columns found in warehouse schema:", suspicious_cols)

# Confirm the baseline itself only uses current observed fields
used_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

print("Features used by baseline:", used_features)
print("Future-window fields used: none")
print("Product flags used: none")
print("Label-derived fields used: none")

WEAK PICKS


,rank,content_hash_id,reason_code,score,why_weak
1,2,content_fec55986a1868d62,low_ctr,33383,"Only one observed signal triggered, so the evi..."
2,3,content_44f34c0a90047651,low_ctr,32958,"Only one observed signal triggered, so the evi..."
3,4,content_fec55986a1868d62,low_ctr,31472,"Only one observed signal triggered, so the evi..."
4,5,content_9c057b66c30a3abb,low_ctr,28973,"Only one observed signal triggered, so the evi..."
5,6,content_9c057b66c30a3abb,low_ctr,28947,"Only one observed signal triggered, so the evi..."
6,7,content_34a70fea29d15f24,low_ctr,27410,"Only one observed signal triggered, so the evi..."
7,8,content_66288edeb93b7c4f,poor_position,24577,"Only one observed signal triggered, so the evi..."
8,9,content_9c057b66c30a3abb,low_ctr,24233,"Only one observed signal triggered, so the evi..."
9,10,content_66288edeb93b7c4f,poor_position,23542,"Only one observed signal triggered, so the evi..."
10,11,content_757b1fa67827358d,low_ctr,19301,"Only one observed signal triggered, so the evi..."



LEAKAGE CHECK
Suspicious columns found in warehouse schema: []
Features used by baseline: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
Future-window fields used: none
Product flags used: none
Label-derived fields used: none


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.